# Compare Infomap and Leiden in a Scanpy workflow

This tutorial shows how to run Infomap on an AnnData neighbor graph and compare the result with Scanpy's Leiden workflow. It uses a small synthetic dataset to avoid network downloads and to keep the example reproducible.

Infomap reads the sparse observation graph from `adata.obsp` and writes categorical labels to `adata.obs`, matching Scanpy `tl` conventions. Leiden remains the standard Scanpy baseline shown here; if the optional Leiden dependency is unavailable, the notebook keeps running and reports that comparison as skipped.


In [ ]:
import importlib.util

SCANPY_AVAILABLE = importlib.util.find_spec("scanpy") is not None
if not SCANPY_AVAILABLE:
    print('Install scanpy to run the full workflow: python -m pip install "scanpy[leiden]"')
else:
    import infomap
    import numpy as np
    import pandas as pd
    import scanpy as sc
    from sklearn.datasets import make_blobs

    print("infomap:", infomap.__version__)
    print("scanpy:", sc.__version__)
    print("pandas:", pd.__version__)


## Create a small AnnData object

The dataset is synthetic and local. Scanpy builds a nearest-neighbor graph in `adata.obsp["connectivities"]`, which is the same graph used by Scanpy clustering tools and the default graph read by `infomap.tl.infomap()`.


In [ ]:
if SCANPY_AVAILABLE:
    X, truth = make_blobs(
        n_samples=120,
        centers=4,
        n_features=12,
        cluster_std=1.8,
        random_state=123,
    )
    adata = sc.AnnData(X)
    adata.obs["truth"] = pd.Categorical([str(label) for label in truth])

    sc.pp.neighbors(adata, n_neighbors=12, random_state=123)
    sc.tl.umap(adata, random_state=123)

    adata


## Run Infomap and Leiden

`infomap.tl.infomap()` stores labels in `adata.obs[key_added]` and run metadata in `adata.uns[key_added]`. Scanpy's Leiden function is called with the same neighbor graph. Leiden is skipped gracefully if the optional backend is not installed.


In [ ]:
if SCANPY_AVAILABLE:
    infomap.tl.infomap(
        adata,
        key_added="infomap",
        seed=123,
        num_trials=20,
    )

    leiden_note = None
    try:
        sc.tl.leiden(adata, key_added="leiden", random_state=123)
    except Exception as exc:
        leiden_note = f"Scanpy Leiden skipped: {exc}"

    print("Infomap communities:", adata.obs["infomap"].nunique())
    if "leiden" in adata.obs:
        print("Leiden communities:", adata.obs["leiden"].nunique())
    else:
        print(leiden_note)

    adata.uns["infomap"]


## Compare assignments

The labels are categorical strings because that is the common Scanpy representation for cluster assignments.


In [ ]:
if SCANPY_AVAILABLE:
    columns = ["truth", "infomap"] + (["leiden"] if "leiden" in adata.obs else [])
    comparison = adata.obs[columns].copy()
    comparison.head(10)


In [ ]:
if SCANPY_AVAILABLE:
    try:
        from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
    except ImportError:
        print("Install scikit-learn to compute ARI and NMI.")
    else:
        rows = [
            {
                "method": "infomap",
                "ARI vs truth": adjusted_rand_score(comparison["truth"], comparison["infomap"]),
                "NMI vs truth": normalized_mutual_info_score(comparison["truth"], comparison["infomap"]),
            }
        ]
        if "leiden" in comparison:
            rows.append(
                {
                    "method": "leiden",
                    "ARI vs truth": adjusted_rand_score(comparison["truth"], comparison["leiden"]),
                    "NMI vs truth": normalized_mutual_info_score(comparison["truth"], comparison["leiden"]),
                }
            )
        pd.DataFrame(rows)


## Visualize the clusters

When Scanpy plotting is available, color the same UMAP layout by the known synthetic labels and the detected communities.


In [ ]:
if SCANPY_AVAILABLE:
    colors = ["truth", "infomap"] + (["leiden"] if "leiden" in adata.obs else [])
    sc.pl.umap(adata, color=colors, wspace=0.35)


## Notes on graph choices

By default, Infomap uses `adata.obsp["connectivities"]`. Pass `neighbors_key` when using a named Scanpy neighbor graph, or `obsp`/`adjacency` when selecting a graph directly. Use `directed=True` for directed observation graphs and `use_weights=False` for unweighted treatment of nonzero sparse entries.

If you use Infomap in published work, cite the Infomap software and the map equation literature. See the repository and Infomap user guide for the current recommended references.
